# 03 — Resultados ejecutivos de scoring | RADAR Cibest

**Fase ASUM-DM:** 4 — Modelado  
**Responsable:** Jhon Farley Adarve Díaz  
**Fecha de ejecución:** Junio 2026  
**Propósito:** ejecutar, auditar e interpretar el scoring híbrido RADAR Cibest, integrando TOPSIS, Índice de Proximidad con Colombia, Trend macroeconómico, rankings por línea, bandas de atractividad, gaps, drivers y restricciones.

---

## 0. El score RADAR debe interpretarse como una tesis de atractividad, no como un ranking aislado

Este notebook responde la pregunta central del modelo:

> **¿Qué países presentan mayor atractividad relativa para la internacionalización de Grupo Cibest, considerando fundamentos estructurales, proximidad con Colombia y momentum macroeconómico reciente?**

El scoring RADAR combina tres componentes:

```text
RADAR = alpha * TOPSIS + beta * IPC + gamma * Trend
```

Donde:

- **TOPSIS** mide atractivo estructural país-variable.
- **IPC** mide proximidad con Colombia.
- **Trend** captura momentum macro reciente a partir de `gdp_growth`.

### Mensaje ejecutivo preliminar

El output principal no debe leerse como una lista mecánica de posiciones. Debe interpretarse por:

1. score y ranking RADAR;
2. ranking TOPSIS estructural;
3. contribución de IPC y Trend;
4. bandas de atractividad;
5. gaps y empates prácticos;
6. drivers y restricciones por país/línea;
7. diferencias de tesis entre líneas de negocio.

---

## 1. Configuración del entorno y estilo Cibest

In [28]:
# ---------------------------------------------------------------------------
# Configuración inicial del notebook
# ---------------------------------------------------------------------------
import sys
from pathlib import Path
import importlib
import re
import warnings

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import HTML, display, Markdown

warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path.cwd().parent))

# ---------------------------------------------------------------------------
# Recarga controlada de módulos del proyecto
# ---------------------------------------------------------------------------

import sys
import importlib
import inspect
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import src.data_preparation.cleaning as cleaning
import src.data_preparation.feature_engineering as feature_engineering
import src.scoring.hybrid_scorer as hybrid_scorer
import src.scoring.ranking as ranking
import src.scoring.explainability as explainability
import src.scoring.monte_carlo as monte_carlo

importlib.invalidate_caches()

# Orden importante: primero dependencias, luego módulos que las importan
importlib.reload(cleaning)
importlib.reload(feature_engineering)
importlib.reload(ranking)
importlib.reload(explainability)
importlib.reload(monte_carlo)
importlib.reload(hybrid_scorer)

from src.utils import (
    load_all_configs,
    resolve_data_path,
    setup_logger,
    get_variable_catalog,
    get_world_bank_variable_catalog,
)

from src.scoring.hybrid_scorer import (
    run_full_scoring,
    prepare_decision_matrix,
    _build_business_line_weights,
)

from src.scoring.explainability import (
    compute_all_business_line_contributions,
    build_explainability_table_for_line,
    get_top_contributors,
    get_top_shortfalls,
    summarize_contributions_by_dimension,
    generate_country_line_explanation,
    compare_country_across_lines,
    compare_countries_in_line,
    compute_all_marginal_effects,
    combine_contribution_and_marginal,
    classify_driver_robustness,
    build_country_driver_table,
)

configs = load_all_configs()
setup_logger(configs["settings"].get("logging"))
variable_catalog = get_variable_catalog(configs["variables"])

# ---------------------------------------------------------------------------
# Identidad visual Cibest
# ---------------------------------------------------------------------------
CIBEST = {
    "gray": "#2C2A28",
    "gray_light": "#CCCAC7",
    "yellow": "#FDD923",
    "gold": "#D6B302",
    "gold_light": "#FFF7D3",
    "gold_dark": "#8F7701",
    "gray_bg": "#F5F5F5",
    "gray_border": "#D0D0D0",
    "white": "#FFFFFF",
    "green": "#0BA682",
    "amber": "#FF7E41",
    "red": "#C62828",
}

TIER_COLORS = {
    "Alta": CIBEST["green"],
    "Media-alta": CIBEST["gold"],
    "Media": CIBEST["amber"],
    "Baja": CIBEST["red"],
}

px.defaults.template = dict(
    layout=dict(
        font=dict(family="Arial, sans-serif", size=13, color=CIBEST["gray"]),
        title=dict(font=dict(size=17, color=CIBEST["gray"])),
        plot_bgcolor=CIBEST["white"],
        paper_bgcolor=CIBEST["white"],
        xaxis=dict(gridcolor=CIBEST["gray_border"], linecolor=CIBEST["gray"]),
        yaxis=dict(gridcolor=CIBEST["gray_border"], linecolor=CIBEST["gray"]),
        colorway=[CIBEST["gray"], CIBEST["gold"], CIBEST["green"], CIBEST["amber"]],
    )
)


def style_table(df, gradient_cols=None, gradient_cmap="YlGnBu", format_dict=None):
    """Aplica estilo Cibest a tablas pandas."""
    styler = df.style.set_table_styles([
        {"selector": "th", "props": [
            ("background-color", CIBEST["gray"]),
            ("color", CIBEST["yellow"]),
            ("font-weight", "bold"),
            ("text-align", "center"),
            ("padding", "8px"),
            ("font-family", "Arial, sans-serif"),
        ]},
        {"selector": "td", "props": [
            ("padding", "6px 10px"),
            ("font-family", "Arial, sans-serif"),
            ("border-bottom", f"1px solid {CIBEST['gray_border']}"),
        ]},
    ])
    if gradient_cols:
        styler = styler.background_gradient(subset=gradient_cols, cmap=gradient_cmap)
    if format_dict:
        styler = styler.format(format_dict)
    return styler


def insight_box(title: str, text: str):
    display(HTML(f"""
    <div style="border-left:6px solid {CIBEST['gold']}; background-color:{CIBEST['gold_light']};
                padding:14px 18px; margin:12px 0; font-family:Arial, sans-serif; color:{CIBEST['gray']};">
        <b>{title}</b><br>{text}
    </div>
    """))


def risk_box(title: str, text: str):
    display(HTML(f"""
    <div style="border-left:6px solid {CIBEST['red']}; background-color:#FDECEC;
                padding:14px 18px; margin:12px 0; font-family:Arial, sans-serif; color:{CIBEST['gray']};">
        <b>{title}</b><br>{text}
    </div>
    """))


def success_box(title: str, text: str):
    display(HTML(f"""
    <div style="border-left:6px solid {CIBEST['green']}; background-color:#E8F7F3;
                padding:14px 18px; margin:12px 0; font-family:Arial, sans-serif; color:{CIBEST['gray']};">
        <b>{title}</b><br>{text}
    </div>
    """))

success_box("Entorno listo", "Módulos recargados, configuración activa y estilo Cibest disponible para visualización ejecutiva.")

---

## 2. Se carga el master vigente y se valida que sea apto para scoring

In [29]:
raw_dir = resolve_data_path(configs["settings"]["data"]["raw_path"])
pattern = re.compile(r"^master_raw_\d{8}\.parquet$")

master_files = sorted(
    [path for path in raw_dir.glob("master_raw_*.parquet") if pattern.match(path.name)],
    key=lambda path: path.stat().st_mtime,
    reverse=True,
)

if not master_files:
    raise FileNotFoundError("Falta master_raw_YYYYMMDD.parquet. Ejecute primero notebook 01.")

master_path = master_files[0]
master = pd.read_parquet(master_path)

required_cols = {"country_iso3", "year", "variable", "value", "source"}
missing_cols = required_cols - set(master.columns)

if missing_cols:
    raise ValueError(f"Master inválido. Faltan columnas: {missing_cols}")

master["country_iso3"] = master["country_iso3"].astype(str).str.strip()
master["variable"] = master["variable"].astype(str).str.strip()

if "gdp_growth" not in master["variable"].unique():
    raise ValueError("El master no contiene gdp_growth. Trend no podrá calcularse correctamente.")

min_expected_variables = int(configs["settings"].get("data_quality", {}).get("min_expected_variables", 35))
if master["variable"].nunique() < min_expected_variables:
    raise ValueError(
        f"Master parcial: solo {master['variable'].nunique()} variables. "
        f"Mínimo esperado: {min_expected_variables}."
    )

master = master[master["variable"] != "wgi_composite"].copy()

master_summary = pd.DataFrame({
    "métrica": ["Archivo", "Filas", "Países", "Variables", "Tiene gdp_growth"],
    "valor": [
        master_path.name,
        master.shape[0],
        master["country_iso3"].nunique(),
        master["variable"].nunique(),
        "Sí" if "gdp_growth" in master["variable"].unique() else "No",
    ],
})

display(style_table(master_summary))

,métrica,valor
0,Archivo,master_raw_20260616.parquet
1,Filas,1288
2,Países,30
3,Variables,45
4,Tiene gdp_growth,Sí


**Lectura ejecutiva.** El scoring solo debe ejecutarse si el master contiene el universo de variables esperado, incluye `gdp_growth` para Trend y no corresponde a una extracción parcial.

---

## 3. La matriz de decisión excluye variables que no deben entrar a TOPSIS

In [30]:
wide_raw, decision_matrix, excluded = prepare_decision_matrix(master, configs)

matrix_checks = pd.DataFrame({
    "control": [
        "wide_raw filas",
        "wide_raw variables",
        "decision_matrix filas",
        "decision_matrix variables",
        "gdp_growth en master",
        "gdp_growth en wide_raw",
        "gdp_growth en decision_matrix",
        "wgi_composite en decision_matrix",
    ],
    "resultado": [
        wide_raw.shape[0],
        wide_raw.shape[1],
        decision_matrix.shape[0],
        decision_matrix.shape[1],
        "Sí" if "gdp_growth" in master["variable"].unique() else "No",
        "Sí" if "gdp_growth" in wide_raw.columns else "No",
        "Sí" if "gdp_growth" in decision_matrix.columns else "No",
        "Sí" if "wgi_composite" in decision_matrix.columns else "No",
    ],
})

display(style_table(matrix_checks))

if "gdp_growth" in decision_matrix.columns:
    raise ValueError("gdp_growth está en decision_matrix. Debe excluirse de TOPSIS y usarse solo en Trend.")

2026-06-22 16:03:01 | INFO     | src.data_preparation.cleaning:pivot_latest_value_and_year:133 | Pivoteo ancho: 30 paises x 45 variables (estrategia=latest_available)
2026-06-22 16:03:01 | INFO     | src.data_preparation.cleaning:apply_freshness_filter:200 | Control de antigüedad aplicado: reference_year=2026, max_age=6, celdas stale=51
2026-06-22 16:03:01 | INFO     | src.data_preparation.cleaning:apply_freshness_filter:225 | Variables con más datos stale: {'public_debt_gdp': 8, 'bank_capital_rwa': 6, 'bank_concentration_5': 5, 'stock_market_cap_gdp': 5, 'financial_system_deposits_gdp': 4, 'interest_rate_spread': 4, 'atms_per_100k_adults': 3, 'current_account_gdp': 2, 'ict_goods_exports_pct_total_goods_exports': 2, 'domestic_credit_private_gdp': 2, 'digital_payment_adults_pct': 2, 'personal_remittances_gdp': 2, 'trade_openness': 2, 'commercial_bank_branches_per_100k_adults': 1, 'gdp_per_capita_ppp': 1}
2026-06-22 16:03:01 | INFO     | src.data_preparation.cleaning:apply_freshness_filt

,control,resultado
0,wide_raw filas,25
1,wide_raw variables,46
2,decision_matrix filas,25
3,decision_matrix variables,35
4,gdp_growth en master,Sí
5,gdp_growth en wide_raw,Sí
6,gdp_growth en decision_matrix,No
7,wgi_composite en decision_matrix,No


**Interpretación.** `wide_raw` conserva variables necesarias para IPC, Trend, auditoría y dashboard. `decision_matrix` debe contener únicamente variables elegibles para TOPSIS. `gdp_growth` debe estar en `master` y `wide_raw`, pero no en `decision_matrix`.

---

## 4. El scoring completo integra TOPSIS, IPC, Trend y RADAR

In [31]:
results = run_full_scoring(master, configs, persist=True)

scoring_summary = pd.DataFrame({
    "métrica": [
        "Países evaluados",
        "Países excluidos por cobertura",
        "País origen",
        "País origen excluido",
        "Alpha TOPSIS",
        "Beta IPC",
        "Gamma Trend",
    ],
    "valor": [
        len(results["radar_global"]),
        ", ".join(results["excluded_countries"]) if results["excluded_countries"] else "Ninguno",
        results["origin_country"],
        results["origin_country_excluded"],
        results["composite_weights"]["alpha"],
        results["composite_weights"]["beta"],
        results["composite_weights"]["gamma"],
    ],
})

display(style_table(scoring_summary))

2026-06-22 16:03:01 | INFO     | src.data_preparation.cleaning:pivot_latest_value_and_year:133 | Pivoteo ancho: 30 paises x 45 variables (estrategia=latest_available)
2026-06-22 16:03:01 | INFO     | src.data_preparation.cleaning:apply_freshness_filter:200 | Control de antigüedad aplicado: reference_year=2026, max_age=6, celdas stale=51
2026-06-22 16:03:01 | INFO     | src.data_preparation.cleaning:apply_freshness_filter:225 | Variables con más datos stale: {'public_debt_gdp': 8, 'bank_capital_rwa': 6, 'bank_concentration_5': 5, 'stock_market_cap_gdp': 5, 'financial_system_deposits_gdp': 4, 'interest_rate_spread': 4, 'atms_per_100k_adults': 3, 'current_account_gdp': 2, 'ict_goods_exports_pct_total_goods_exports': 2, 'domestic_credit_private_gdp': 2, 'digital_payment_adults_pct': 2, 'personal_remittances_gdp': 2, 'trade_openness': 2, 'commercial_bank_branches_per_100k_adults': 1, 'gdp_per_capita_ppp': 1}
2026-06-22 16:03:01 | INFO     | src.data_preparation.cleaning:apply_freshness_filt

,métrica,valor
0,Países evaluados,24
1,Países excluidos por cobertura,"BRB, CUB, GUY, HTI, VEN"
2,País origen,COL
3,País origen excluido,True
4,Alpha TOPSIS,0.600000
5,Beta IPC,0.300000
6,Gamma Trend,0.100000


**Lectura ejecutiva.** El score RADAR usa la combinación configurada de TOPSIS estructural, IPC y Trend. El país origen debe excluirse del universo evaluado para evitar comparaciones no válidas.

---

## 5. El ranking RADAR global muestra la prioridad final bajo la fórmula compuesta

In [32]:
radar_global = results["radar_global"].copy()

if "country_iso3" not in radar_global.columns:
    radar_global = radar_global.reset_index().rename(columns={"index": "country_iso3"})

radar_global = radar_global.sort_values("radar_score", ascending=False).copy()
radar_global["rank"] = radar_global["radar_score"].rank(ascending=False, method="min").astype(int)

display(style_table(
    radar_global.head(15),
    gradient_cols=["radar_score"],
    gradient_cmap="RdYlGn",
    format_dict={"radar_score": "{:.3f}", "rank": "{:.0f}"},
))

,country_iso3,radar_score,rank
0,PAN,0.691,1
1,CRI,0.653,2
2,ESP,0.627,3
3,DOM,0.620,4
4,CHL,0.569,5
5,USA,0.552,6
6,URY,0.539,7
7,ECU,0.538,8
8,MEX,0.533,9
9,GTM,0.530,10


In [33]:
fig = px.bar(
    radar_global.head(15).sort_values("radar_score"),
    x="radar_score",
    y="country_iso3",
    orientation="h",
    title="Top 15 países por score RADAR global",
    color="radar_score",
    color_continuous_scale=[[0, CIBEST["gray_light"]], [1, CIBEST["gold"]]],
)
fig.update_layout(xaxis_title="Score RADAR", yaxis_title="País")
fig.show()

**Interpretación.** Este ranking es la lectura final del modelo híbrido global. Sin embargo, no reemplaza la lectura por línea de negocio: cada línea tiene una tesis de atractividad distinta.

---

## 6. TOPSIS global representa el atractivo estructural antes de proximidad y tendencia

In [34]:
global_ranking = results["global_ranking"].copy()

if "country_iso3" not in global_ranking.columns:
    global_ranking = global_ranking.reset_index().rename(columns={"index": "country_iso3"})

display(style_table(
    global_ranking.head(15),
    gradient_cols=["score"],
    gradient_cmap="RdYlGn",
    format_dict={"score": "{:.3f}", "rank": "{:.0f}"},
))

,country_iso3,score,d_pos,d_neg,score_macro,score_financial,score_institutional,score_digital_tech,rank
0,USA,0.703,0.065024,0.153921,0.690735,0.648331,0.786068,0.716415,1
1,CAN,0.693,0.064999,0.146931,0.684788,0.573351,0.905093,0.634003,2
2,ESP,0.644,0.073321,0.132823,0.603907,0.604259,0.725187,0.711742,3
3,CHL,0.609,0.081121,0.126179,0.519289,0.553528,0.763021,0.645377,4
4,URY,0.580,0.086752,0.119684,0.454172,0.492884,0.807951,0.607994,5
5,CRI,0.544,0.092541,0.110278,0.450534,0.481363,0.708748,0.546621,6
6,BHS,0.528,0.095968,0.107182,0.386003,0.557674,0.651762,0.506325,7
7,PAN,0.513,0.096542,0.101864,0.459363,0.528073,0.543181,0.531762,8
8,JAM,0.500,0.101475,0.101457,0.370086,0.578026,0.569446,0.452131,9
9,BRA,0.500,0.103998,0.103839,0.535240,0.509160,0.449603,0.537484,10


**Lectura ejecutiva.** TOPSIS responde quién tiene mejores fundamentos estructurales. RADAR puede alterar el orden al incorporar proximidad con Colombia y momentum macro.

---

## 7. IPC y Trend explican movimientos entre TOPSIS y RADAR

In [35]:
ipc = results["ipc"].copy()
trend = results["trend"].copy()

if "country_iso3" not in ipc.columns:
    ipc = ipc.reset_index().rename(columns={"index": "country_iso3"})
if "country_iso3" not in trend.columns:
    trend = trend.reset_index().rename(columns={"index": "country_iso3"})

ipc_display = ipc[["country_iso3", "ipc"]].sort_values("ipc", ascending=False)
trend_display = trend[["country_iso3", "trend"]].sort_values("trend", ascending=False)

display(style_table(ipc_display.head(15), gradient_cols=["ipc"], gradient_cmap="RdYlGn", format_dict={"ipc": "{:.3f}"}))
display(style_table(trend_display.head(15), gradient_cols=["trend"], gradient_cmap="RdYlGn", format_dict={"trend": "{:.3f}"}))

,country_iso3,ipc
0,ECU,0.954
1,PAN,0.942
2,DOM,0.864
3,CRI,0.835
4,PER,0.777
5,SLV,0.749
6,BOL,0.736
7,GTM,0.735
8,HND,0.724
9,NIC,0.712


,country_iso3,trend
16,PAN,1.000
1,BHS,1.000
7,CRI,0.763
2,BLZ,0.702
8,DOM,0.629
10,ESP,0.621
15,NIC,0.560
11,GTM,0.544
12,HND,0.536
4,BRA,0.402


**Interpretación.** IPC identifica países con mayor proximidad relativa a Colombia. Trend identifica países con mayor momentum macro reciente. Un país puede subir en RADAR aunque no sea top estructural si presenta alta proximidad o tendencia favorable.

---

## 8. La descomposición RADAR muestra qué componente mueve el ranking final

In [36]:
# TOPSIS
base_topsis = results["global_ranking"].copy()
if "country_iso3" in base_topsis.columns:
    topsis = base_topsis.set_index("country_iso3")["score"].rename("topsis_score")
else:
    topsis = base_topsis["score"].rename("topsis_score")

# IPC
ipc_series = ipc.set_index("country_iso3")["ipc"].rename("ipc")

# Trend
trend_series = trend.set_index("country_iso3")["trend"].rename("trend")

component_df = pd.concat([topsis, ipc_series, trend_series], axis=1)
component_df["ipc"] = component_df["ipc"].fillna(component_df["ipc"].median())
component_df["trend"] = component_df["trend"].fillna(component_df["trend"].median())

alpha = results["composite_weights"]["alpha"]
beta = results["composite_weights"]["beta"]
gamma = results["composite_weights"]["gamma"]

component_df["aporte_topsis"] = alpha * component_df["topsis_score"]
component_df["aporte_ipc"] = beta * component_df["ipc"]
component_df["aporte_trend"] = gamma * component_df["trend"]
component_df["radar_score_recalc"] = component_df[["aporte_topsis", "aporte_ipc", "aporte_trend"]].sum(axis=1)
component_df["rank_topsis"] = component_df["topsis_score"].rank(ascending=False, method="min").astype(int)
component_df["rank_radar"] = component_df["radar_score_recalc"].rank(ascending=False, method="min").astype(int)
component_df["delta_rank"] = component_df["rank_topsis"] - component_df["rank_radar"]
component_df = component_df.sort_values("rank_radar")

component_display = component_df.reset_index().rename(columns={"index": "country_iso3"})

display(style_table(
    component_display.head(20),
    gradient_cols=["radar_score_recalc", "delta_rank"],
    gradient_cmap="RdYlGn",
    format_dict={
        "topsis_score": "{:.3f}",
        "ipc": "{:.3f}",
        "trend": "{:.3f}",
        "aporte_topsis": "{:.3f}",
        "aporte_ipc": "{:.3f}",
        "aporte_trend": "{:.3f}",
        "radar_score_recalc": "{:.3f}",
        "rank_topsis": "{:.0f}",
        "rank_radar": "{:.0f}",
        "delta_rank": "{:+.0f}",
    },
))

,country_iso3,topsis_score,ipc,trend,aporte_topsis,aporte_ipc,aporte_trend,radar_score_recalc,rank_topsis,rank_radar,delta_rank
0,PAN,0.513,0.942,1.000,0.308,0.283,0.100,0.691,8,1,+7
1,CRI,0.544,0.835,0.763,0.326,0.250,0.076,0.653,6,2,+4
2,ESP,0.644,0.596,0.621,0.387,0.179,0.062,0.627,3,3,+0
3,DOM,0.497,0.864,0.629,0.298,0.259,0.063,0.620,11,4,+7
4,CHL,0.609,0.668,0.037,0.365,0.200,0.004,0.569,4,5,-1
5,USA,0.703,0.341,0.278,0.422,0.102,0.028,0.552,1,6,-5
6,URY,0.580,0.540,0.290,0.348,0.162,0.029,0.539,5,7,-2
7,ECU,0.405,0.954,0.082,0.243,0.286,0.008,0.538,20,8,+12
8,MEX,0.487,0.703,0.303,0.292,0.211,0.030,0.533,13,9,+4
9,GTM,0.424,0.735,0.544,0.255,0.221,0.054,0.530,18,10,+8


In [37]:
component_means = component_df[["aporte_topsis", "aporte_ipc", "aporte_trend"]].mean().reset_index()
component_means.columns = ["componente", "aporte_promedio"]

fig = px.bar(
    component_means,
    x="componente",
    y="aporte_promedio",
    title="TOPSIS domina el score RADAR por diseño, pero IPC y Trend explican cambios marginales",
    color="componente",
    color_discrete_sequence=[CIBEST["gray"], CIBEST["gold"], CIBEST["green"]],
)
fig.update_layout(showlegend=False, yaxis_title="Aporte promedio")
fig.show()

**Implicación.** Esta descomposición permite explicar por qué países estructuralmente fuertes pueden bajar si tienen baja proximidad, o por qué países con fundamentos medios pueden subir por IPC o Trend.

---

## 9. Rankings por línea: cada línea expresa una tesis de atractividad distinta

In [38]:
for business_line, ranking_df in results["business_line_rankings"].items():
    tmp = ranking_df.copy()
    if "country_iso3" not in tmp.columns:
        tmp = tmp.reset_index().rename(columns={"index": "country_iso3"})

    display(Markdown(f"### {business_line} — Top 15"))
    display(style_table(
        tmp.sort_values("rank").head(15),
        gradient_cols=["score"],
        gradient_cmap="RdYlGn",
        format_dict={"score": "{:.3f}", "rank": "{:.0f}"},
    ))

### IB — Top 15

,country_iso3,score,d_pos,d_neg,score_macro,score_financial,score_institutional,score_digital_tech,rank,business_line
0,USA,0.728,0.069187,0.185379,0.602323,0.724245,0.806634,0.755174,1,IB
1,ESP,0.702,0.077198,0.181436,0.569905,0.710840,0.730944,0.736835,2,IB
2,CAN,0.690,0.079777,0.177210,0.700119,0.612076,0.923344,0.668285,3,IB
3,CHL,0.652,0.088248,0.165188,0.572716,0.631479,0.740357,0.666526,4,IB
4,BHS,0.605,0.093983,0.143697,0.490480,0.624528,0.608889,0.549265,5,IB
5,URY,0.589,0.099897,0.143002,0.531180,0.516798,0.774739,0.628212,6,IB
6,CRI,0.566,0.107425,0.140127,0.499425,0.533213,0.675389,0.542714,7,IB
7,PAN,0.551,0.111445,0.136976,0.508816,0.601164,0.447478,0.485130,8,IB
8,JAM,0.551,0.107349,0.131622,0.450975,0.594725,0.499410,0.435583,9,IB
9,BRA,0.549,0.116547,0.141606,0.525375,0.625887,0.362913,0.549839,10,IB


### PF — Top 15

,country_iso3,score,d_pos,d_neg,score_macro,score_financial,score_institutional,score_digital_tech,rank,business_line
0,ESP,0.674,0.106675,0.221031,0.601785,0.414891,0.716088,0.801321,1,PF
1,CAN,0.643,0.123035,0.221196,0.580706,0.388662,0.912720,0.743135,2,PF
2,USA,0.628,0.124665,0.210155,0.368215,0.392141,0.800436,0.772037,3,PF
3,CHL,0.599,0.126681,0.188863,0.557462,0.349608,0.739730,0.702810,4,PF
4,URY,0.557,0.133233,0.167780,0.449576,0.316646,0.772727,0.647786,5,PF
5,BRA,0.543,0.139214,0.165422,0.374663,0.356690,0.375422,0.640003,6,PF
6,ARG,0.533,0.140726,0.160459,0.240726,0.339840,0.401102,0.647275,7,PF
7,CRI,0.509,0.138643,0.143486,0.603669,0.347756,0.685725,0.543971,8,PF
8,JAM,0.492,0.146294,0.141481,0.531548,0.759893,0.515285,0.414473,9,PF
9,SUR,0.470,0.149501,0.132568,0.418761,0.479647,0.317696,0.478239,10,PF


### AD — Top 15

,country_iso3,score,d_pos,d_neg,score_macro,score_financial,score_institutional,score_digital_tech,rank,business_line
0,USA,0.801,0.063790,0.256111,0.596469,0.714881,0.863337,0.735109,1,AD
1,CAN,0.743,0.089247,0.258114,0.730481,0.674982,0.957619,0.559761,2,AD
2,ESP,0.652,0.109739,0.205961,0.588407,0.564732,0.737571,0.551622,3,AD
3,CHL,0.637,0.116333,0.204145,0.593937,0.594103,0.764572,0.483999,4,AD
4,URY,0.618,0.124808,0.202261,0.549772,0.485928,0.777732,0.418258,5,AD
5,CRI,0.552,0.142952,0.176033,0.513842,0.302899,0.698506,0.352634,6,AD
6,BHS,0.515,0.151733,0.160935,0.581412,0.590891,0.559446,0.432479,7,AD
7,BLZ,0.475,0.178088,0.161049,0.458552,0.343613,0.360680,0.602588,8,AD
8,PAN,0.470,0.162813,0.144577,0.542400,0.442576,0.428784,0.532170,9,AD
9,ARG,0.409,0.179976,0.124298,0.339557,0.382140,0.402441,0.421955,10,AD


### BD — Top 15

,country_iso3,score,d_pos,d_neg,score_macro,score_financial,score_institutional,score_digital_tech,rank,business_line
0,USA,0.798,0.051639,0.203797,0.812200,0.872102,0.797886,0.756028,1,BD
1,ESP,0.745,0.064699,0.189119,0.664895,0.783640,0.715501,0.784127,2,BD
2,CAN,0.724,0.071361,0.187164,0.713779,0.765925,0.911762,0.688799,3,BD
3,CHL,0.703,0.073650,0.174381,0.601183,0.733342,0.739814,0.759375,4,BD
4,BRA,0.638,0.091245,0.160721,0.714709,0.702259,0.375171,0.582621,5,BD
5,URY,0.622,0.094600,0.155996,0.502265,0.601487,0.774467,0.724425,6,BD
6,ARG,0.622,0.093949,0.154760,0.581802,0.610284,0.401832,0.705888,7,BD
7,CRI,0.582,0.100652,0.139986,0.474197,0.609311,0.686019,0.636604,8,BD
8,MEX,0.539,0.114324,0.133799,0.710308,0.428174,0.335169,0.510803,9,BD
9,BHS,0.531,0.124507,0.141059,0.366582,0.644911,0.600528,0.609106,10,BD


### CIB — Top 15

,country_iso3,score,d_pos,d_neg,score_macro,score_financial,score_institutional,score_digital_tech,rank,business_line
0,CAN,0.731,0.073278,0.199112,0.658005,0.719947,0.940030,0.512759,1,CIB
1,USA,0.705,0.088390,0.211475,0.575674,0.778565,0.827973,0.683950,2,CIB
2,ESP,0.641,0.093691,0.167489,0.616682,0.625770,0.743458,0.513287,3,CIB
3,CHL,0.640,0.095425,0.169361,0.540104,0.659406,0.746662,0.459598,4,CIB
4,BHS,0.552,0.117604,0.144714,0.407939,0.626575,0.596803,0.412426,5,CIB
5,JAM,0.525,0.123679,0.136954,0.375096,0.626710,0.492826,0.281199,6,CIB
6,TTO,0.514,0.127550,0.134741,0.378126,0.626959,0.410912,0.260698,7,CIB
7,DOM,0.508,0.128981,0.133010,0.446482,0.544867,0.482710,0.283899,8,CIB
8,URY,0.504,0.129284,0.131465,0.394837,0.476197,0.769076,0.409129,9,CIB
9,BRA,0.504,0.131899,0.134043,0.524212,0.536593,0.355436,0.395872,10,CIB


**Lectura ejecutiva.** El modelo no produce cinco rankings independientes sin relación; produce cinco lecturas del atractivo país según tesis de negocio. `IB` y `CIB` tienden a privilegiar profundidad financiera, escala e institucionalidad. `PF` y `BD` privilegian adopción digital, flujos, canales e inclusión. `AD` opera como tesis digital-institucional.

---

## 10. La auditoría de pesos verifica que cada línea usa pesos efectivos y overrides esperados

In [39]:
from typing import Any, Dict


def audit_business_line_weights(
    configs: Dict[str, Dict[str, Any]],
    decision_matrix: pd.DataFrame,
) -> pd.DataFrame:
    """Audita pesos efectivos usados por TOPSIS para cada línea de negocio."""

    business_lines = configs["business_lines"]["business_lines"]
    global_dim_weights = configs["weights"]["dimension_weights"]
    global_variable_weights = configs["weights"]["variable_weights"]

    rows = []

    for bl_key, bl_cfg in business_lines.items():
        dim_weights_line, final_var_weights = _build_business_line_weights(
            business_line_cfg=bl_cfg,
            variable_weights_by_dim=global_variable_weights,
        )

        final_var_weights_filtered = {
            var: weight
            for var, weight in final_var_weights.items()
            if var in decision_matrix.columns
        }

        total_filtered = sum(final_var_weights_filtered.values())
        if total_filtered > 0:
            final_var_weights_filtered = {
                var: weight / total_filtered
                for var, weight in final_var_weights_filtered.items()
            }

        overrides = bl_cfg.get("variable_weight_overrides", {}) or {}

        for dim, vars_global in global_variable_weights.items():
            dim_weight_line = dim_weights_line.get(dim, 0.0)
            dim_weight_global = global_dim_weights.get(dim)

            for var, global_var_weight in vars_global.items():
                override_weight = None
                if dim in overrides and var in overrides[dim]:
                    override_weight = overrides[dim][var]

                rows.append({
                    "business_line": bl_key,
                    "dimension": dim,
                    "variable": var,
                    "in_decision_matrix": var in decision_matrix.columns,
                    "global_dimension_weight": dim_weight_global,
                    "line_dimension_weight": dim_weight_line,
                    "global_variable_weight_in_dim": global_var_weight,
                    "override_variable_weight_in_dim": override_weight,
                    "has_override": override_weight is not None,
                    "final_topsis_weight": final_var_weights_filtered.get(var, 0.0),
                })

    return (
        pd.DataFrame(rows)
        .sort_values(["business_line", "dimension", "final_topsis_weight"], ascending=[True, True, False])
        .reset_index(drop=True)
    )

weights_audit = audit_business_line_weights(configs=configs, decision_matrix=decision_matrix)

weight_sum_check = (
    weights_audit[weights_audit["in_decision_matrix"]]
    .groupby("business_line")["final_topsis_weight"]
    .sum()
    .round(6)
    .reset_index()
)

display(style_table(weight_sum_check, format_dict={"final_topsis_weight": "{:.6f}"}))

,business_line,final_topsis_weight
0,AD,1.000000
1,BD,1.000000
2,CIB,1.000000
3,IB,1.000000
4,PF,1.000000


In [40]:
inactive_weighted_vars = (
    weights_audit[~weights_audit["in_decision_matrix"]]
    [["business_line", "dimension", "variable", "global_variable_weight_in_dim"]]
    .drop_duplicates()
)

display(style_table(inactive_weighted_vars))

,business_line,dimension,variable,global_variable_weight_in_dim


**Interpretación.** Los pesos finales deben sumar 1 por línea después de filtrar variables presentes en `decision_matrix`. Si aparecen variables ponderadas pero ausentes, se requiere corregir configuración, extracción o matriz de decisión.

---

## 11. El spread de pesos muestra qué variables diferencian realmente las líneas

In [41]:
weight_matrix = (
    weights_audit[weights_audit["in_decision_matrix"]]
    .pivot_table(
        index="variable",
        columns="business_line",
        values="final_topsis_weight",
        aggfunc="sum",
        fill_value=0.0,
    )
)

weight_matrix["spread"] = weight_matrix.max(axis=1) - weight_matrix.min(axis=1)

weight_spread = weight_matrix.sort_values("spread", ascending=False).reset_index()

display(style_table(
    weight_spread.head(25),
    gradient_cols=["spread"],
    gradient_cmap="YlOrRd",
    format_dict={"spread": "{:.4f}"},
))

business_line,variable,AD,BD,CIB,IB,PF,spread
0,digital_payment_adults_pct,0.026400,0.054000,0.002000,0.008000,0.188000,0.1860
1,stock_market_cap_gdp,0.036000,0.002500,0.144000,0.013500,0.007500,0.1415
2,regulatory_quality,0.163200,0.033000,0.055200,0.063000,0.027600,0.1356
3,secure_internet_servers_per_million,0.132000,0.048000,0.006000,0.012000,0.023500,0.1260
4,rule_of_law,0.124800,0.027000,0.062400,0.069000,0.021600,0.1032
5,internet_users_pct,0.046200,0.102000,0.003400,0.011000,0.051700,0.0986
6,personal_remittances_gdp,0.007200,0.002500,0.016000,0.004500,0.095000,0.0925
7,domestic_credit_private_gdp,0.013200,0.035000,0.064000,0.099000,0.017500,0.0858
8,account_ownership,0.006000,0.090000,0.008000,0.054000,0.055000,0.0840
9,mobile_subscriptions,0.023100,0.066000,0.001600,0.009000,0.084600,0.0830


**Implicación.** Las variables con mayor `spread` son las que explican la diferenciación metodológica entre líneas. Deben ser conceptualmente coherentes con la tesis de negocio de cada línea.

---

## 12. La correlación entre líneas identifica familias de tesis de negocio

In [42]:
rankings = {}

for business_line, ranking_df in results["business_line_rankings"].items():
    tmp = ranking_df.copy()
    if "country_iso3" in tmp.columns:
        tmp = tmp.set_index("country_iso3")
    rankings[business_line] = tmp["rank"]

rank_df = pd.DataFrame(rankings)
rank_corr = rank_df.corr(method="spearman")

display(style_table(rank_corr, gradient_cols=rank_corr.columns.tolist(), gradient_cmap="YlOrRd", format_dict={col: "{:.3f}" for col in rank_corr.columns}))

,IB,PF,AD,BD,CIB
IB,1.000,0.697,0.783,0.643,0.777
PF,0.697,1.000,0.863,0.885,0.637
AD,0.783,0.863,1.000,0.838,0.717
BD,0.643,0.885,0.838,1.000,0.731
CIB,0.777,0.637,0.717,0.731,1.000


In [43]:
fig = px.imshow(
    rank_corr,
    text_auto=".2f",
    color_continuous_scale=[[0, CIBEST["green"]], [0.5, CIBEST["gold"]], [1, CIBEST["red"]]],
    title="Las correlaciones entre líneas revelan familias de atractividad",
)
fig.update_layout(height=550)
fig.show()

In [44]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# weights_audit debe venir del notebook 03
weight_vectors = (
    weights_audit[weights_audit["in_decision_matrix"]]
    .pivot_table(
        index="business_line",
        columns="variable",
        values="final_topsis_weight",
        aggfunc="sum",
        fill_value=0.0,
    )
)

weight_cosine = pd.DataFrame(
    cosine_similarity(weight_vectors),
    index=weight_vectors.index,
    columns=weight_vectors.index,
)

display(
    style_table(
        weight_cosine,
        gradient_cols=weight_cosine.columns.tolist(),
        gradient_cmap="YlOrRd",
        format_dict={col: "{:.3f}" for col in weight_cosine.columns},
    ).set_caption("Similitud coseno entre vectores de pesos por línea")
)

business_line,AD,BD,CIB,IB,PF
business_line,,,,,
AD,1.000,0.490,0.500,0.564,0.370
BD,0.490,1.000,0.409,0.610,0.601
CIB,0.500,0.409,1.000,0.720,0.285
IB,0.564,0.610,0.720,1.000,0.365
PF,0.370,0.601,0.285,0.365,1.000


Interpretación:
- Alta similitud de pesos + alta correlación de ranking:
    el problema está en configuración.

- Baja similitud de pesos + alta correlación de ranking:
    los países tienen estructura subyacente común.

In [45]:
#ver qué variables explican la similitud 
def compare_weight_vectors(
    weight_vectors: pd.DataFrame,
    line_a: str,
    line_b: str,
    top_n: int = 20,
) -> pd.DataFrame:
    out = pd.DataFrame({
        "variable": weight_vectors.columns,
        f"weight_{line_a}": weight_vectors.loc[line_a].values,
        f"weight_{line_b}": weight_vectors.loc[line_b].values,
    })

    out["abs_diff"] = (
        out[f"weight_{line_a}"] - out[f"weight_{line_b}"]
    ).abs()

    out["min_shared_weight"] = out[
        [f"weight_{line_a}", f"weight_{line_b}"]
    ].min(axis=1)

    return out.sort_values(
        ["min_shared_weight", "abs_diff"],
        ascending=[False, True],
    ).head(top_n)


display(compare_weight_vectors(weight_vectors, "PF", "BD"))
display(compare_weight_vectors(weight_vectors, "AD", "BD"))
display(compare_weight_vectors(weight_vectors, "PF", "AD"))

,variable,weight_PF,weight_BD,abs_diff,min_shared_weight
21,mobile_subscriptions,0.0846,0.0660,0.0186,0.0660
0,account_ownership,0.0550,0.0900,0.0350,0.0550
9,digital_payment_adults_pct,0.1880,0.0540,0.1340,0.0540
20,internet_users_pct,0.0517,0.1020,0.0503,0.0517
26,regulatory_quality,0.0276,0.0330,0.0054,0.0276
28,secure_internet_servers_per_million,0.0235,0.0480,0.0245,0.0235
18,inflation_rate,0.0224,0.0360,0.0136,0.0224
27,rule_of_law,0.0216,0.0270,0.0054,0.0216
15,government_effectiveness,0.0180,0.0225,0.0045,0.0180
4,bank_npl_ratio,0.0175,0.0275,0.0100,0.0175


,variable,weight_AD,weight_BD,abs_diff,min_shared_weight
28,secure_internet_servers_per_million,0.1320,0.0480,0.0840,0.0480
20,internet_users_pct,0.0462,0.1020,0.0558,0.0462
26,regulatory_quality,0.1632,0.0330,0.1302,0.0330
27,rule_of_law,0.1248,0.0270,0.0978,0.0270
9,digital_payment_adults_pct,0.0264,0.0540,0.0276,0.0264
21,mobile_subscriptions,0.0231,0.0660,0.0429,0.0231
15,government_effectiveness,0.0480,0.0225,0.0255,0.0225
6,control_of_corruption,0.0864,0.0195,0.0669,0.0195
23,political_stability,0.0336,0.0180,0.0156,0.0180
14,gdp_per_capita_ppp,0.0161,0.0420,0.0259,0.0161


,variable,weight_PF,weight_AD,abs_diff,min_shared_weight
20,internet_users_pct,0.0517,0.0462,0.0055,0.0462
26,regulatory_quality,0.0276,0.1632,0.1356,0.0276
9,digital_payment_adults_pct,0.1880,0.0264,0.1616,0.0264
28,secure_internet_servers_per_million,0.0235,0.1320,0.1085,0.0235
21,mobile_subscriptions,0.0846,0.0231,0.0615,0.0231
27,rule_of_law,0.0216,0.1248,0.1032,0.0216
17,ict_goods_exports_pct_total_goods_exports,0.0188,0.0825,0.0637,0.0188
15,government_effectiveness,0.0180,0.0480,0.0300,0.0180
2,bank_capital_rwa,0.0175,0.0180,0.0005,0.0175
6,control_of_corruption,0.0156,0.0864,0.0708,0.0156


Si las mismas variables dominan entre lineas ahí esta la causa.

In [46]:
def topn_overlap(rankings_by_line, n=5):
    def _country_iso3(series_or_df):
        df = series_or_df
        if "country_iso3" in df.columns:
            return df["country_iso3"].astype(str)
        if df.index.name == "country_iso3":
            return df.index.to_series().astype(str)
        reset = df.reset_index()
        if "country_iso3" in reset.columns:
            return reset["country_iso3"].astype(str)
        return reset.iloc[:, 0].astype(str)

    rows = []
    lines = list(rankings_by_line.keys())

    for i, line_a in enumerate(lines):
        for line_b in lines[i + 1:]:
            a = (
                rankings_by_line[line_a]
                .assign(country_iso3=_country_iso3(rankings_by_line[line_a]))
                .sort_values("rank")
                .head(n)["country_iso3"]
                .tolist()
            )
            b = (
                rankings_by_line[line_b]
                .assign(country_iso3=_country_iso3(rankings_by_line[line_b]))
                .sort_values("rank")
                .head(n)["country_iso3"]
                .tolist()
            )

            overlap = len(set(a) & set(b))

            rows.append({
                "line_a": line_a,
                "line_b": line_b,
                "n": n,
                "overlap": overlap,
                "overlap_pct": overlap / n,
                "top_a": ", ".join(a),
                "top_b": ", ".join(b),
            })

    return pd.DataFrame(rows).sort_values("overlap_pct", ascending=False)

display(topn_overlap(results["business_line_rankings"], n=10))
display(topn_overlap(results["business_line_rankings"], n=10))

,line_a,line_b,n,overlap,overlap_pct,top_a,top_b
0,IB,PF,10,8,0.8,"USA, ESP, CAN, CHL, BHS, URY, CRI, PAN, JAM, BRA","ESP, CAN, USA, CHL, URY, BRA, ARG, CRI, JAM, SUR"
1,IB,AD,10,8,0.8,"USA, ESP, CAN, CHL, BHS, URY, CRI, PAN, JAM, BRA","USA, CAN, ESP, CHL, URY, CRI, BHS, BLZ, PAN, ARG"
2,IB,BD,10,8,0.8,"USA, ESP, CAN, CHL, BHS, URY, CRI, PAN, JAM, BRA","USA, ESP, CAN, CHL, BRA, URY, ARG, CRI, MEX, BHS"
3,IB,CIB,10,8,0.8,"USA, ESP, CAN, CHL, BHS, URY, CRI, PAN, JAM, BRA","CAN, USA, ESP, CHL, BHS, JAM, TTO, DOM, URY, BRA"
5,PF,BD,10,8,0.8,"ESP, CAN, USA, CHL, URY, BRA, ARG, CRI, JAM, SUR","USA, ESP, CAN, CHL, BRA, URY, ARG, CRI, MEX, BHS"
7,AD,BD,10,8,0.8,"USA, CAN, ESP, CHL, URY, CRI, BHS, BLZ, PAN, ARG","USA, ESP, CAN, CHL, BRA, URY, ARG, CRI, MEX, BHS"
4,PF,AD,10,7,0.7,"ESP, CAN, USA, CHL, URY, BRA, ARG, CRI, JAM, SUR","USA, CAN, ESP, CHL, URY, CRI, BHS, BLZ, PAN, ARG"
6,PF,CIB,10,7,0.7,"ESP, CAN, USA, CHL, URY, BRA, ARG, CRI, JAM, SUR","CAN, USA, ESP, CHL, BHS, JAM, TTO, DOM, URY, BRA"
9,BD,CIB,10,7,0.7,"USA, ESP, CAN, CHL, BRA, URY, ARG, CRI, MEX, BHS","CAN, USA, ESP, CHL, BHS, JAM, TTO, DOM, URY, BRA"
8,AD,CIB,10,6,0.6,"USA, CAN, ESP, CHL, URY, CRI, BHS, BLZ, PAN, ARG","CAN, USA, ESP, CHL, BHS, JAM, TTO, DOM, URY, BRA"


,line_a,line_b,n,overlap,overlap_pct,top_a,top_b
0,IB,PF,10,8,0.8,"USA, ESP, CAN, CHL, BHS, URY, CRI, PAN, JAM, BRA","ESP, CAN, USA, CHL, URY, BRA, ARG, CRI, JAM, SUR"
1,IB,AD,10,8,0.8,"USA, ESP, CAN, CHL, BHS, URY, CRI, PAN, JAM, BRA","USA, CAN, ESP, CHL, URY, CRI, BHS, BLZ, PAN, ARG"
2,IB,BD,10,8,0.8,"USA, ESP, CAN, CHL, BHS, URY, CRI, PAN, JAM, BRA","USA, ESP, CAN, CHL, BRA, URY, ARG, CRI, MEX, BHS"
3,IB,CIB,10,8,0.8,"USA, ESP, CAN, CHL, BHS, URY, CRI, PAN, JAM, BRA","CAN, USA, ESP, CHL, BHS, JAM, TTO, DOM, URY, BRA"
5,PF,BD,10,8,0.8,"ESP, CAN, USA, CHL, URY, BRA, ARG, CRI, JAM, SUR","USA, ESP, CAN, CHL, BRA, URY, ARG, CRI, MEX, BHS"
7,AD,BD,10,8,0.8,"USA, CAN, ESP, CHL, URY, CRI, BHS, BLZ, PAN, ARG","USA, ESP, CAN, CHL, BRA, URY, ARG, CRI, MEX, BHS"
4,PF,AD,10,7,0.7,"ESP, CAN, USA, CHL, URY, BRA, ARG, CRI, JAM, SUR","USA, CAN, ESP, CHL, URY, CRI, BHS, BLZ, PAN, ARG"
6,PF,CIB,10,7,0.7,"ESP, CAN, USA, CHL, URY, BRA, ARG, CRI, JAM, SUR","CAN, USA, ESP, CHL, BHS, JAM, TTO, DOM, URY, BRA"
9,BD,CIB,10,7,0.7,"USA, ESP, CAN, CHL, BRA, URY, ARG, CRI, MEX, BHS","CAN, USA, ESP, CHL, BHS, JAM, TTO, DOM, URY, BRA"
8,AD,CIB,10,6,0.6,"USA, CAN, ESP, CHL, URY, CRI, BHS, BLZ, PAN, ARG","CAN, USA, ESP, CHL, BHS, JAM, TTO, DOM, URY, BRA"


**Lectura ejecutiva.** Una correlación alta no implica error. Puede indicar que dos líneas comparten fundamentos país. La meta no es forzar rankings independientes, sino que cada línea responda a una tesis de negocio defendible.

---

## 13. Bandas, gaps y empates prácticos evitan sobreinterpretar rankings finos

In [47]:
def classify_gap(gap: float) -> str:
    if pd.isna(gap):
        return "Sin siguiente país"
    if gap < 0.005:
        return "Empate práctico"
    if gap < 0.015:
        return "Diferencia débil"
    if gap < 0.030:
        return "Diferencia moderada"
    return "Diferencia material"


def assign_tier_by_score(score: float, q80: float, q60: float, q40: float) -> str:
    if score >= q80:
        return "Alta"
    if score >= q60:
        return "Media-alta"
    if score >= q40:
        return "Media"
    return "Baja"


def strategic_read(row: pd.Series) -> str:
    tier = row["attractiveness_tier"]
    gap = row["gap_interpretation"]

    if tier == "Alta" and gap == "Diferencia material":
        return "Liderazgo o posicionamiento claramente diferenciado"
    if tier == "Alta" and gap in ["Empate práctico", "Diferencia débil"]:
        return "Alta atractividad, pero no distinguible ordinalmente del país siguiente"
    if tier == "Media-alta" and gap == "Empate práctico":
        return "Banda competitiva media-alta; decisión requiere análisis de drivers"
    if tier == "Media":
        return "Atractividad intermedia; priorizar solo si hay racional estratégico específico"
    return "Lectura dependiente de drivers y restricciones de implementación"


tier_tables = {}

for business_line, ranking_df in results["business_line_rankings"].items():
    tmp = ranking_df.copy()
    if "country_iso3" in tmp.columns:
        tmp = tmp.set_index("country_iso3")

    tmp = tmp.sort_values("score", ascending=False).copy()

    q80 = tmp["score"].quantile(0.80)
    q60 = tmp["score"].quantile(0.60)
    q40 = tmp["score"].quantile(0.40)

    tmp["attractiveness_tier"] = tmp["score"].apply(lambda x: assign_tier_by_score(x, q80, q60, q40))
    tmp["score_gap_next"] = tmp["score"] - tmp["score"].shift(-1)
    tmp["gap_interpretation"] = tmp["score_gap_next"].apply(classify_gap)
    tmp["strategic_read"] = tmp.apply(strategic_read, axis=1)

    tier_tables[business_line] = tmp[["score", "rank", "attractiveness_tier", "score_gap_next", "gap_interpretation", "strategic_read"]]

for business_line, table in tier_tables.items():
    display(Markdown(f"### {business_line} — bandas y gaps"))
    display(style_table(
        table.head(15).reset_index(),
        gradient_cols=["score", "score_gap_next"],
        gradient_cmap="RdYlGn",
        format_dict={"score": "{:.3f}", "rank": "{:.0f}", "score_gap_next": "{:.3f}"},
    ))

### IB — bandas y gaps

,country_iso3,score,rank,attractiveness_tier,score_gap_next,gap_interpretation,strategic_read
0,USA,0.728,1,Alta,0.027,Diferencia moderada,Lectura dependiente de drivers y restricciones de implementación
1,ESP,0.702,2,Alta,0.012,Diferencia débil,"Alta atractividad, pero no distinguible ordinalmente del país siguiente"
2,CAN,0.690,3,Alta,0.038,Diferencia material,Liderazgo o posicionamiento claramente diferenciado
3,CHL,0.652,4,Alta,0.047,Diferencia material,Liderazgo o posicionamiento claramente diferenciado
4,BHS,0.605,5,Alta,0.016,Diferencia moderada,Lectura dependiente de drivers y restricciones de implementación
5,URY,0.589,6,Media-alta,0.023,Diferencia moderada,Lectura dependiente de drivers y restricciones de implementación
6,CRI,0.566,7,Media-alta,0.015,Diferencia débil,Lectura dependiente de drivers y restricciones de implementación
7,PAN,0.551,8,Media-alta,0.001,Empate práctico,Banda competitiva media-alta; decisión requiere análisis de drivers
8,JAM,0.551,9,Media-alta,0.002,Empate práctico,Banda competitiva media-alta; decisión requiere análisis de drivers
9,BRA,0.549,10,Media-alta,0.004,Empate práctico,Banda competitiva media-alta; decisión requiere análisis de drivers


### PF — bandas y gaps

,country_iso3,score,rank,attractiveness_tier,score_gap_next,gap_interpretation,strategic_read
0,ESP,0.674,1,Alta,0.032,Diferencia material,Liderazgo o posicionamiento claramente diferenciado
1,CAN,0.643,2,Alta,0.015,Diferencia débil,"Alta atractividad, pero no distinguible ordinalmente del país siguiente"
2,USA,0.628,3,Alta,0.029,Diferencia moderada,Lectura dependiente de drivers y restricciones de implementación
3,CHL,0.599,4,Alta,0.041,Diferencia material,Liderazgo o posicionamiento claramente diferenciado
4,URY,0.557,5,Alta,0.014,Diferencia débil,"Alta atractividad, pero no distinguible ordinalmente del país siguiente"
5,BRA,0.543,6,Media-alta,0.010,Diferencia débil,Lectura dependiente de drivers y restricciones de implementación
6,ARG,0.533,7,Media-alta,0.024,Diferencia moderada,Lectura dependiente de drivers y restricciones de implementación
7,CRI,0.509,8,Media-alta,0.017,Diferencia moderada,Lectura dependiente de drivers y restricciones de implementación
8,JAM,0.492,9,Media-alta,0.022,Diferencia moderada,Lectura dependiente de drivers y restricciones de implementación
9,SUR,0.470,10,Media-alta,0.002,Empate práctico,Banda competitiva media-alta; decisión requiere análisis de drivers


### AD — bandas y gaps

,country_iso3,score,rank,attractiveness_tier,score_gap_next,gap_interpretation,strategic_read
0,USA,0.801,1,Alta,0.058,Diferencia material,Liderazgo o posicionamiento claramente diferenciado
1,CAN,0.743,2,Alta,0.091,Diferencia material,Liderazgo o posicionamiento claramente diferenciado
2,ESP,0.652,3,Alta,0.015,Diferencia moderada,Lectura dependiente de drivers y restricciones de implementación
3,CHL,0.637,4,Alta,0.019,Diferencia moderada,Lectura dependiente de drivers y restricciones de implementación
4,URY,0.618,5,Alta,0.067,Diferencia material,Liderazgo o posicionamiento claramente diferenciado
5,CRI,0.552,6,Media-alta,0.037,Diferencia material,Lectura dependiente de drivers y restricciones de implementación
6,BHS,0.515,7,Media-alta,0.040,Diferencia material,Lectura dependiente de drivers y restricciones de implementación
7,BLZ,0.475,8,Media-alta,0.005,Empate práctico,Banda competitiva media-alta; decisión requiere análisis de drivers
8,PAN,0.470,9,Media-alta,0.062,Diferencia material,Lectura dependiente de drivers y restricciones de implementación
9,ARG,0.409,10,Media-alta,0.004,Empate práctico,Banda competitiva media-alta; decisión requiere análisis de drivers


### BD — bandas y gaps

,country_iso3,score,rank,attractiveness_tier,score_gap_next,gap_interpretation,strategic_read
0,USA,0.798,1,Alta,0.053,Diferencia material,Liderazgo o posicionamiento claramente diferenciado
1,ESP,0.745,2,Alta,0.021,Diferencia moderada,Lectura dependiente de drivers y restricciones de implementación
2,CAN,0.724,3,Alta,0.021,Diferencia moderada,Lectura dependiente de drivers y restricciones de implementación
3,CHL,0.703,4,Alta,0.065,Diferencia material,Liderazgo o posicionamiento claramente diferenciado
4,BRA,0.638,5,Alta,0.015,Diferencia moderada,Lectura dependiente de drivers y restricciones de implementación
5,URY,0.622,6,Media-alta,0.000,Empate práctico,Banda competitiva media-alta; decisión requiere análisis de drivers
6,ARG,0.622,7,Media-alta,0.041,Diferencia material,Lectura dependiente de drivers y restricciones de implementación
7,CRI,0.582,8,Media-alta,0.042,Diferencia material,Lectura dependiente de drivers y restricciones de implementación
8,MEX,0.539,9,Media-alta,0.008,Diferencia débil,Lectura dependiente de drivers y restricciones de implementación
9,BHS,0.531,10,Media-alta,0.002,Empate práctico,Banda competitiva media-alta; decisión requiere análisis de drivers


### CIB — bandas y gaps

,country_iso3,score,rank,attractiveness_tier,score_gap_next,gap_interpretation,strategic_read
0,CAN,0.731,1,Alta,0.026,Diferencia moderada,Lectura dependiente de drivers y restricciones de implementación
1,USA,0.705,2,Alta,0.064,Diferencia material,Liderazgo o posicionamiento claramente diferenciado
2,ESP,0.641,3,Alta,0.002,Empate práctico,"Alta atractividad, pero no distinguible ordinalmente del país siguiente"
3,CHL,0.640,4,Alta,0.088,Diferencia material,Liderazgo o posicionamiento claramente diferenciado
4,BHS,0.552,5,Alta,0.026,Diferencia moderada,Lectura dependiente de drivers y restricciones de implementación
5,JAM,0.525,6,Media-alta,0.012,Diferencia débil,Lectura dependiente de drivers y restricciones de implementación
6,TTO,0.514,7,Media-alta,0.006,Diferencia débil,Lectura dependiente de drivers y restricciones de implementación
7,DOM,0.508,8,Media-alta,0.004,Empate práctico,Banda competitiva media-alta; decisión requiere análisis de drivers
8,URY,0.504,9,Media-alta,0.000,Empate práctico,Banda competitiva media-alta; decisión requiere análisis de drivers
9,BRA,0.504,10,Media-alta,0.028,Diferencia moderada,Lectura dependiente de drivers y restricciones de implementación


**Implicación.** Cuando los gaps son bajos, las diferencias de posición deben comunicarse como empates prácticos o bandas, no como jerarquías finas.

---

## 14. La dispersión entre líneas muestra países sensibles a la tesis de negocio

In [48]:
rank_cols = ["IB", "PF", "AD", "BD", "CIB"]
rank_df_clean = rank_df[rank_cols].copy()

rank_df_clean["rank_std_across_lines"] = rank_df_clean[rank_cols].std(axis=1)
rank_df_clean["rank_range_across_lines"] = rank_df_clean[rank_cols].max(axis=1) - rank_df_clean[rank_cols].min(axis=1)

rank_dispersion = rank_df_clean.sort_values("rank_range_across_lines", ascending=False)

display(style_table(
    rank_dispersion.head(20).reset_index(),
    gradient_cols=["rank_range_across_lines", "rank_std_across_lines"],
    gradient_cmap="YlOrRd",
    format_dict={"rank_std_across_lines": "{:.2f}", "rank_range_across_lines": "{:.0f}"},
))

,country_iso3,IB,PF,AD,BD,CIB,rank_std_across_lines,rank_range_across_lines
0,ARG,20,7,10,7,22,7.26,15
1,MEX,23,19,16,9,12,5.54,14
2,BLZ,13,16,8,20,17,4.55,12
3,SUR,22,10,18,17,21,4.72,12
4,CRI,7,8,6,8,18,4.88,12
5,BOL,12,21,23,18,16,4.30,11
6,TTO,11,18,15,16,7,4.39,11
7,BHS,5,14,7,10,5,3.83,9
8,DOM,17,12,11,11,8,3.27,9
9,HND,15,22,22,23,19,3.27,8


**Lectura ejecutiva.** Países con alto rango entre líneas no son “inestables”; son países cuya atractividad depende fuertemente de la tesis de negocio. Estos casos requieren análisis comercial específico.

---

## 15. Contribuciones ponderadas explican drivers y restricciones de forma ejecutiva

> Nota metodológica: `contribution = normalized_value * final_topsis_weight` no es una descomposición exacta de TOPSIS, porque TOPSIS depende de distancias al ideal positivo y negativo. Es una capa de explicabilidad ejecutiva post-normalización.

In [49]:
contrib_by_line = compute_all_business_line_contributions(
    decision_matrix=decision_matrix,
    weights_audit=weights_audit,
)

pf_explainability = build_explainability_table_for_line(
    ranking_df=results["business_line_rankings"]["PF"],
    contributions=contrib_by_line["PF"],
    top_n=3,
)

display(style_table(pf_explainability.head(15), format_dict={"score": "{:.3f}", "rank": "{:.0f}"}))

,country_iso3,score,rank,top_drivers,top_constraints
0,ESP,0.674,1,digital_payment_adults_pct; account_ownership; internet_users_pct,personal_remittances_gdp; mobile_subscriptions; atms_per_100k_adults
1,CAN,0.643,2,digital_payment_adults_pct; account_ownership; internet_users_pct,personal_remittances_gdp; mobile_subscriptions; trade_openness
2,USA,0.628,3,digital_payment_adults_pct; account_ownership; internet_users_pct,personal_remittances_gdp; trade_openness; mobile_subscriptions
3,CHL,0.599,4,digital_payment_adults_pct; internet_users_pct; mobile_subscriptions,personal_remittances_gdp; atms_per_100k_adults; mobile_subscriptions
4,URY,0.557,5,digital_payment_adults_pct; atms_per_100k_adults; mobile_subscriptions,personal_remittances_gdp; digital_payment_adults_pct; trade_openness
5,BRA,0.543,6,digital_payment_adults_pct; account_ownership; atms_per_100k_adults,personal_remittances_gdp; mobile_subscriptions; digital_payment_adults_pct
6,ARG,0.533,7,digital_payment_adults_pct; mobile_subscriptions; internet_users_pct,personal_remittances_gdp; digital_payment_adults_pct; trade_openness
7,CRI,0.509,8,digital_payment_adults_pct; mobile_subscriptions; internet_users_pct,digital_payment_adults_pct; personal_remittances_gdp; atms_per_100k_adults
8,JAM,0.492,9,personal_remittances_gdp; digital_payment_adults_pct; internet_users_pct,digital_payment_adults_pct; atms_per_100k_adults; mobile_subscriptions
9,SUR,0.470,10,digital_payment_adults_pct; mobile_subscriptions; personal_remittances_gdp,digital_payment_adults_pct; atms_per_100k_adults; personal_remittances_gdp


In [50]:
country_focus = "ARG"
line_focus = "PF"

display(style_table(get_top_contributors(contrib_by_line[line_focus], country_focus, top_n=8), gradient_cols=["contribution"], gradient_cmap="RdYlGn", format_dict={"normalized_value": "{:.3f}", "final_topsis_weight": "{:.4f}", "contribution": "{:.4f}"}))
display(style_table(get_top_shortfalls(contrib_by_line[line_focus], country_focus, top_n=8), gradient_cols=["shortfall"], gradient_cmap="YlOrRd", format_dict={"normalized_value": "{:.3f}", "final_topsis_weight": "{:.4f}", "shortfall": "{:.4f}"}))

print(generate_country_line_explanation(contrib_by_line[line_focus], country_focus, top_n=3))

,business_line,country_iso3,dimension,variable,normalized_value,final_topsis_weight,contribution
750,PF,ARG,digital_tech,digital_payment_adults_pct,0.681,0.1880,0.1280
725,PF,ARG,digital_tech,mobile_subscriptions,0.668,0.0846,0.0565
700,PF,ARG,digital_tech,internet_users_pct,0.836,0.0517,0.0432
300,PF,ARG,financial,account_ownership,0.778,0.0550,0.0428
825,PF,ARG,digital_tech,atms_per_100k_adults,0.477,0.0611,0.0291
475,PF,ARG,financial,bank_capital_rwa,1.000,0.0175,0.0175
800,PF,ARG,digital_tech,commercial_bank_branches_per_100k_adults,0.351,0.0423,0.0149
325,PF,ARG,financial,interest_rate_spread,0.838,0.0150,0.0126


,business_line,country_iso3,dimension,variable,normalized_value,final_topsis_weight,shortfall
400,PF,ARG,financial,personal_remittances_gdp,0.037,0.0950,0.0915
750,PF,ARG,digital_tech,digital_payment_adults_pct,0.681,0.1880,0.0600
200,PF,ARG,macro,trade_openness,0.031,0.0512,0.0496
825,PF,ARG,digital_tech,atms_per_100k_adults,0.477,0.0611,0.0320
725,PF,ARG,digital_tech,mobile_subscriptions,0.668,0.0846,0.0281
800,PF,ARG,digital_tech,commercial_bank_branches_per_100k_adults,0.351,0.0423,0.0274
50,PF,ARG,macro,inflation_rate,0.000,0.0224,0.0224
850,PF,ARG,digital_tech,ict_goods_exports_pct_total_goods_exports,0.000,0.0188,0.0188


ARG en PF está impulsado principalmente por digital_payment_adults_pct (0.128), mobile_subscriptions (0.057), internet_users_pct (0.043). Sus principales restricciones relativas son personal_remittances_gdp (0.091), digital_payment_adults_pct (0.060), trade_openness (0.050).


**Interpretación.** Los drivers positivos explican qué variables elevan el score estructural de un país. Las restricciones muestran dónde el país pierde atractivo relativo frente al ideal.

---

## 16. Comparar países y líneas permite convertir ranking en narrativa estratégica

In [51]:
# País entre líneas: por qué un país cambia según tesis de negocio
compare_country = compare_country_across_lines(
    contrib_by_line=contrib_by_line,
    country_iso3="ARG",
    top_n=5,
)

display(style_table(compare_country, gradient_cols=["contribution"], gradient_cmap="RdYlGn", format_dict={"normalized_value": "{:.3f}", "final_topsis_weight": "{:.4f}", "contribution": "{:.4f}", "shortfall": "{:.4f}"}))

,business_line,country_iso3,dimension,variable,normalized_value,final_topsis_weight,contribution,shortfall
0,AD,ARG,digital_tech,secure_internet_servers_per_million,0.486,0.1320,0.0642,0.0678
1,AD,ARG,institutional,regulatory_quality,0.390,0.1632,0.0636,0.0996
2,AD,ARG,institutional,rule_of_law,0.444,0.1248,0.0554,0.0694
3,AD,ARG,digital_tech,internet_users_pct,0.836,0.0462,0.0386,0.0076
4,AD,ARG,institutional,control_of_corruption,0.361,0.0864,0.0312,0.0552
5,BD,ARG,digital_tech,internet_users_pct,0.836,0.1020,0.0853,0.0167
6,BD,ARG,financial,account_ownership,0.778,0.0900,0.0700,0.0200
7,BD,ARG,macro,population_total,0.702,0.0840,0.0590,0.0250
8,BD,ARG,macro,urban_population_pct,0.938,0.0480,0.0450,0.0030
9,BD,ARG,digital_tech,mobile_subscriptions,0.668,0.0660,0.0441,0.0219


In [52]:
# Países dentro de una línea: por qué un líder supera al bloque siguiente
compare_pf_top = compare_countries_in_line(
    contributions=contrib_by_line["PF"],
    countries=["ESP", "USA", "CAN", "CHL"],
    top_n=12,
)

display(style_table(compare_pf_top))

country_iso3,dimension,variable,CAN,CHL,ESP,USA
0,digital_tech,atms_per_100k_adults,0.049691,0.019281,0.032574,0.041197
1,digital_tech,commercial_bank_branches_per_100k_adults,0.021935,0.010537,0.030965,0.027611
2,digital_tech,digital_payment_adults_pct,0.188000,0.155745,0.186122,0.175655
3,digital_tech,internet_users_pct,0.049745,0.051467,0.051700,0.050219
4,digital_tech,mobile_subscriptions,0.020837,0.050667,0.048811,0.035603
5,digital_tech,secure_internet_servers_per_million,0.017085,0.013657,0.016285,0.021639
6,financial,account_ownership,0.055000,0.045212,0.054982,0.053986
7,institutional,regulatory_quality,0.027600,0.021466,0.019780,0.027497
8,institutional,rule_of_law,0.021600,0.016203,0.017890,0.018101
9,macro,inflation_rate,0.022116,0.021921,0.022076,0.022058


**Implicación.** Esta capa permite explicar diferencias como: un país puede ser competitivo en `PF` y `BD`, pero débil en `IB` o `CIB`, no por error del modelo, sino porque cada línea pondera tesis diferentes.

---

## 17. El análisis marginal valida si un driver es robusto o solo descriptivo

El análisis marginal leave-one-variable-out calcula:

```text
score_effect = score_full - score_without_variable
```

Interpretación:

- `score_effect > 0`: la variable ayuda al país.
- `score_effect < 0`: la variable penaliza al país.
- `abs(score_effect)` alto: variable material.
- `abs(score_effect)` bajo: variable poco decisiva.

In [53]:
marginal_by_line = compute_all_marginal_effects(
    decision_matrix=decision_matrix,
    weights_audit=weights_audit,
    variable_catalog=variable_catalog,
    distance_metric=configs["settings"]["scoring"]["topsis"].get("distance_metric", "euclidean"),
)

pf_marginal = marginal_by_line["PF"]

line_variable_importance = (
    pf_marginal
    .groupby(["business_line", "dimension", "removed_variable"], as_index=False)
    .agg(
        mean_abs_effect=("abs_score_effect", "mean"),
        mean_effect=("score_effect", "mean"),
        max_abs_effect=("abs_score_effect", "max"),
    )
    .sort_values("mean_abs_effect", ascending=False)
)

display(style_table(
    line_variable_importance.head(20),
    gradient_cols=["mean_abs_effect"],
    gradient_cmap="YlOrRd",
    format_dict={"mean_abs_effect": "{:.4f}", "mean_effect": "{:.4f}", "max_abs_effect": "{:.4f}"},
))

2026-06-22 16:03:03 | INFO     | src.scoring.ranking:rank:133 | TOPSIS completado: 25 paises | score max=0.801 (USA)
2026-06-22 16:03:03 | INFO     | src.scoring.ranking:rank:133 | TOPSIS completado: 25 paises | score max=0.801 (USA)
2026-06-22 16:03:03 | INFO     | src.scoring.ranking:rank:133 | TOPSIS completado: 25 paises | score max=0.800 (USA)
2026-06-22 16:03:03 | INFO     | src.scoring.ranking:rank:133 | TOPSIS completado: 25 paises | score max=0.800 (USA)
2026-06-22 16:03:03 | INFO     | src.scoring.ranking:rank:133 | TOPSIS completado: 25 paises | score max=0.801 (USA)
2026-06-22 16:03:03 | INFO     | src.scoring.ranking:rank:133 | TOPSIS completado: 25 paises | score max=0.801 (USA)
2026-06-22 16:03:03 | INFO     | src.scoring.ranking:rank:133 | TOPSIS completado: 25 paises | score max=0.801 (USA)
2026-06-22 16:03:03 | INFO     | src.scoring.ranking:rank:133 | TOPSIS completado: 25 paises | score max=0.801 (USA)
2026-06-22 16:03:03 | INFO     | src.scoring.ranking:rank:133 | 

,business_line,dimension,removed_variable,mean_abs_effect,mean_effect,max_abs_effect
2,PF,digital_tech,digital_payment_adults_pct,0.0711,0.0022,0.1659
14,PF,financial,personal_remittances_gdp,0.0526,-0.0115,0.1107
5,PF,digital_tech,mobile_subscriptions,0.0174,0.0022,0.0600
0,PF,digital_tech,atms_per_100k_adults,0.0080,-0.0038,0.0228
4,PF,digital_tech,internet_users_pct,0.0079,0.0057,0.0182
31,PF,macro,trade_openness,0.0078,-0.0003,0.0216
7,PF,financial,account_ownership,0.0053,0.0038,0.0120
1,PF,digital_tech,commercial_bank_branches_per_100k_adults,0.0041,-0.0015,0.0208
28,PF,macro,inflation_rate,0.0033,0.0031,0.0054
3,PF,digital_tech,ict_goods_exports_pct_total_goods_exports,0.0018,-0.0014,0.0029


**Advertencia metodológica.** No se deben sumar efectos marginales como una descomposición contable del score. TOPSIS es no lineal y depende de distancias al ideal y renormalización de pesos.

---

## 18. Combinar contribución y marginal permite clasificar drivers robustos

In [54]:
pf_combined_explainability = combine_contribution_and_marginal(
    contributions=contrib_by_line["PF"],
    marginal_effects=marginal_by_line["PF"],
)

pf_combined_explainability["driver_class"] = pf_combined_explainability.apply(
    classify_driver_robustness,
    axis=1,
)

arg_driver_table = build_country_driver_table(
    explainability_df=pf_combined_explainability,
    country_iso3="ARG",
    top_n=5,
)

display(style_table(
    arg_driver_table,
    gradient_cols=["contribution", "shortfall", "score_effect"],
    gradient_cmap="RdYlGn",
    format_dict={"normalized_value": "{:.3f}", "contribution": "{:.4f}", "shortfall": "{:.4f}", "score_effect": "{:.4f}", "rank_effect": "{:+.0f}"},
))

,business_line,country_iso3,removed_variable,dimension_contribution,normalized_value,final_topsis_weight_contribution,contribution,shortfall,score_effect,rank_effect,effect_type,driver_side,driver_class
0,PF,ARG,digital_payment_adults_pct,digital_tech,0.681,0.188000,0.1280,0.0600,0.1010,+14,Driver,Driver,Driver robusto
1,PF,ARG,mobile_subscriptions,digital_tech,0.668,0.084600,0.0565,0.0281,0.0114,+0,Driver,Driver,Driver robusto
2,PF,ARG,internet_users_pct,digital_tech,0.836,0.051700,0.0432,0.0085,0.0089,+0,Driver,Driver,Driver descriptivo
3,PF,ARG,account_ownership,financial,0.778,0.055000,0.0428,0.0122,0.0082,+0,Driver,Driver,Driver descriptivo
4,PF,ARG,bank_capital_rwa,financial,1.000,0.017500,0.0175,0.0000,0.0015,+0,Driver,Driver,Efecto marginal bajo
5,PF,ARG,personal_remittances_gdp,financial,0.037,0.095000,0.0035,0.0915,-0.0673,+0,Restriccion,Restriccion,Restriccion critica
6,PF,ARG,trade_openness,macro,0.031,0.051200,0.0016,0.0496,-0.0165,+0,Restriccion,Restriccion,Restriccion critica
7,PF,ARG,commercial_bank_branches_per_100k_adults,digital_tech,0.351,0.042300,0.0149,0.0274,-0.0038,+0,Restriccion,Restriccion,Efecto marginal bajo
8,PF,ARG,inflation_rate,macro,0.000,0.022400,0.0000,0.0224,-0.0032,+0,Restriccion,Restriccion,Efecto marginal bajo
9,PF,ARG,atms_per_100k_adults,digital_tech,0.477,0.061100,0.0291,0.0320,-0.0024,+0,Restriccion,Restriccion,Efecto marginal bajo


**Lectura ejecutiva.** Un driver robusto combina alta contribución y efecto marginal positivo. Una restricción crítica puede tener baja contribución, alta brecha y efecto marginal negativo.

---

## 19. Hallazgos principales

1. RADAR debe leerse como score compuesto: TOPSIS aporta fundamentos estructurales, IPC proximidad y Trend momentum macro.
2. `gdp_growth` debe permanecer fuera de TOPSIS y usarse exclusivamente para Trend, evitando doble conteo de crecimiento.
3. Las líneas de negocio no son rankings independientes; son tesis diferenciadas de atractividad país.
4. La auditoría de pesos permite verificar que overrides y pesos finales cierren correctamente por línea.
5. Las correlaciones entre líneas identifican familias de negocio y similitudes estructurales.
6. Los gaps y bandas son necesarios para no sobrerrepresentar posiciones con diferencias mínimas.
7. La explicabilidad debe combinar contribución ponderada y análisis marginal para distinguir drivers robustos de drivers descriptivos.

---

## 20. Limitaciones

- TOPSIS es relativo al conjunto de países evaluados; agregar o quitar países puede alterar scores.
- Las contribuciones ponderadas no son una descomposición exacta de TOPSIS.
- El análisis marginal es sensibilidad leave-one-variable-out, no causalidad.
- Las bandas por percentiles dependen del universo evaluado.
- RADAR combina componentes con pesos configurados; la robustez de la decisión final debe evaluarse en el notebook 04 mediante Monte Carlo.

---

## 21. Recomendaciones y próximos pasos

1. Presentar rankings por bandas y no solo por posición ordinal.
2. Usar gaps para distinguir líderes robustos de empates prácticos.
3. Reportar drivers y restricciones para países prioritarios por línea.
4. Usar dispersión entre líneas para identificar países dependientes de tesis específica.
5. Validar robustez del ranking RADAR completo en `04_sensitivity_analysis.ipynb` mediante Monte Carlo.
6. Persistir tablas de pesos, bandas, drivers y restricciones como artefactos reproducibles.

---

## 22. Síntesis Ejecutiva

- El scoring RADAR integra atractivo estructural, proximidad y momentum macro en una métrica ejecutiva final.
- Los rankings por línea reflejan tesis de negocio distintas; no deben interpretarse como listas independientes sin contexto.
- Las bandas, gaps y empates prácticos son esenciales para evitar decisiones basadas en diferencias marginales.
- La explicabilidad debe cruzar contribuciones y efectos marginales para identificar drivers realmente robustos.
- El siguiente paso es validar robustez probabilística del RADAR completo en el notebook de sensibilidad.